# Cookbook: Foundry IQ End-to-End

Building production-grade AI systems requires more than simple vector search — you need retrieval that reasons, decomposes complex queries, and synthesizes answers with citations. **Foundry IQ** is Microsoft's intelligence layer that brings retrieval, reasoning, and synthesis together under a single, model-driven experience.

In this cookbook, you will:

- **Create a knowledge source** backed by an Azure AI Search index
- **Create a knowledge base** that pairs your data with an LLM for agentic retrieval
- **Query the knowledge base** and inspect synthesized answers with citations
- **Connect Foundry IQ to the Foundry Agent Service** so an agent can ground its responses in your data

## Prerequisites

You can deploy all required resources automatically using the [Deploy to Azure](https://portal.azure.com/#create/Microsoft.Template/uri/https%3A%2F%2Fraw.githubusercontent.com%2Fmicrosoft%2Fiq-series%2Fmain%2Finfra%2Fazuredeploy.json) button — see the [Episode 1 README](../README.md#-deploy-azure-resources) for step-by-step instructions.

If you prefer to set up resources manually, you need:

1. An **Azure AI Search** service in a [region that supports agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support) with role-based access enabled.
2. A **Microsoft Foundry** project and resource — creating a project automatically provisions the resource.
3. Model deployments: a **text embedding model** (`text-embedding-3-large`) and a **chat model** (`gpt-4o-mini` or equivalent).
4. A **project connection** from your Foundry project to your Azure AI Search service.
5. Appropriate **RBAC roles** on your account: *Search Service Contributor*, *Search Index Data Contributor*, and *Search Index Data Reader* on the search service; *Cognitive Services User* on the Foundry resource for the search service's managed identity.

Create a `.env` file in this directory with your endpoint values:

```
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_EMBEDDING_MODEL=text-embedding-3-large
AOAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
AOAI_GPT_MODEL=gpt-4o-mini
AOAI_GPT_DEPLOYMENT=gpt-4o-mini
FOUNDRY_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_MODEL_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_AI_SEARCH_CONNECTION_NAME=<your-search-connection-name>
```

See the [agentic retrieval quickstart](https://learn.microsoft.com/azure/search/search-get-started-agentic-retrieval) for detailed setup instructions.

## Setup

Install required packages and load environment variables. We authenticate with `DefaultAzureCredential` so no API keys are stored in code.

In [ ]:
%%capture
%pip install -U azure-search-documents==11.6.0b13 azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

# Service endpoints
SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"]
AOAI_ENDPOINT = os.environ["AOAI_ENDPOINT"]

# Model deployments
EMBEDDING_MODEL = os.environ.get("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.environ.get("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.environ.get("AOAI_GPT_MODEL", "gpt-4o-mini")
GPT_DEPLOYMENT = os.environ.get("AOAI_GPT_DEPLOYMENT", "gpt-4o-mini")

# Resource names
INDEX_NAME = "earth-at-night"
KNOWLEDGE_SOURCE_NAME = "earth-knowledge-source"
KNOWLEDGE_BASE_NAME = "earth-knowledge-base"

credential = DefaultAzureCredential()
print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Models: embedding={EMBEDDING_MODEL}, chat={GPT_MODEL}")

We now have authenticated credentials and all configuration loaded from the environment. Nothing is hard-coded.

---

## Step 1 — Create a Search Index and Upload Documents

A knowledge source needs something to search over. We create a search index with text, vector, and semantic search capabilities, then load sample data from NASA's *Earth at Night* e-book. The key requirement for agentic retrieval is a `semantic_search` configuration with a `default_configuration_name`.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SearchField(name="id", type="Edm.String", key=True, filterable=True),
        SearchField(name="page_chunk", type="Edm.String"),
        SearchField(
            name="page_embedding_text_3_large",
            type="Collection(Edm.Single)",
            stored=False,
            vector_search_dimensions=3072,
            vector_search_profile_name="hnsw_text_3_large",
        ),
        SearchField(name="page_number", type="Edm.Int32", filterable=True),
    ],
    vector_search=VectorSearch(
        profiles=[
            VectorSearchProfile(
                name="hnsw_text_3_large",
                algorithm_configuration_name="alg",
                vectorizer_name="azure_openai_text_3_large",
            )
        ],
        algorithms=[HnswAlgorithmConfiguration(name="alg")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_text_3_large",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AOAI_ENDPOINT,
                    deployment_name=EMBEDDING_DEPLOYMENT,
                    model_name=EMBEDDING_MODEL,
                ),
            )
        ],
    ),
    # Semantic config is required for agentic retrieval
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    content_fields=[SemanticField(field_name="page_chunk")]
                ),
            )
        ],
    ),
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
index_client.create_or_update_index(index)
print(f"✅ Index '{INDEX_NAME}' ready")

In [ ]:
import requests
from azure.search.documents import SearchIndexingBufferedSender

DATA_URL = "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/main/nasa-e-book/earth-at-night-json/documents.json"
documents = requests.get(DATA_URL).json()

with SearchIndexingBufferedSender(
    endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential
) as sender:
    sender.upload_documents(documents=documents)

print(f"✅ {len(documents)} documents uploaded to '{INDEX_NAME}'")

We now have a search index populated with chunked text and vector embeddings. This is the raw material that Foundry IQ will reason over.

---

## Step 2 — Create a Knowledge Source

A **knowledge source** tells Foundry IQ *where* to find data. It points at our search index and declares which fields carry source metadata (useful for citations).

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
)

knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="NASA Earth at Night data",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="page_number"),
        ],
    ),
)

index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
print(f"✅ Knowledge source '{KNOWLEDGE_SOURCE_NAME}' ready")

The knowledge source is now registered. It does not store data itself — it is a pointer that the knowledge base will use during retrieval.

---

## Step 3 — Create a Knowledge Base

A **knowledge base** wraps one or more knowledge sources with an LLM configuration. It defines *how* queries are planned, executed, and synthesized. We use `ANSWER_SYNTHESIS` mode so the LLM generates natural-language answers with inline citations instead of returning raw document chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
    KnowledgeRetrievalOutputMode,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AOAI_ENDPOINT,
                deployment_name=GPT_DEPLOYMENT,
                model_name=GPT_MODEL,
            )
        )
    ],
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Provide a concise, informative answer grounded in the retrieved documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"✅ Knowledge base '{KNOWLEDGE_BASE_NAME}' ready")

At this point the full Foundry IQ pipeline is wired: **index → knowledge source → knowledge base → LLM**. We are ready to query.

---

## Step 4 — Query the Knowledge Base

Agentic retrieval goes beyond keyword matching. When you submit a query, the knowledge base:

1. Decomposes the query into focused subqueries
2. Runs them in parallel against the knowledge source
3. Reranks results with the semantic ranker
4. Synthesizes a grounded, cited answer

We pass a conversation-style `messages` list so the pipeline can use context across turns.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    SearchIndexKnowledgeSourceParams,
    KnowledgeRetrievalLowReasoningEffort,
)

retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=credential,
)

query = "What causes city lights to appear brighter from space during the holidays?"

result = retrieval_client.retrieve(
    retrieval_request=KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role="user",
                content=[KnowledgeBaseMessageTextContent(text=query)],
            )
        ],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                include_references=True,
                include_reference_source_data=True,
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort,
    )
)

print("✅ Retrieval complete")

### Inspect the answer

The response contains three key parts:
- **Response** — the synthesized answer
- **Activity** — subqueries and planning steps (useful for debugging)
- **References** — source documents that ground the answer

In [ ]:
import json

# Extract the synthesized answer
answer = "\n\n".join(
    content.text
    for resp in result.response
    for content in resp.content
)
print("📝 Answer:\n", answer)

# Show which subqueries the planner generated
if result.activity:
    print("\n🔍 Query plan:")
    print(json.dumps([a.as_dict() for a in result.activity], indent=2))

# Show source references
if result.references:
    print(f"\n📚 {len(result.references)} reference(s) returned")

Notice how the knowledge base decomposed the question, retrieved relevant chunks, and produced a cited answer — all in a single API call. This is the core value of Foundry IQ: retrieval that *reasons*.

---

## Step 5 — Use with Foundry Agent Service

Foundry IQ becomes even more powerful when combined with the **Foundry Agent Service**. Instead of calling the retrieval API directly, you create an agent that automatically grounds its responses in your search index. The agent handles tool invocation, citation formatting, and multi-turn conversation.

This requires a Foundry project connection to your Azure AI Search service (see [Prerequisites](#prerequisites)).

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAISearchAgentTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
    PromptAgentDefinition,
)

FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["FOUNDRY_MODEL_DEPLOYMENT_NAME"]
SEARCH_CONNECTION = os.environ["AZURE_AI_SEARCH_CONNECTION_NAME"]

project_client = AIProjectClient(
    endpoint=FOUNDRY_ENDPOINT, credential=credential
)

# Resolve the connection ID from the friendly name
connection = project_client.connections.get(SEARCH_CONNECTION)
print(f"✅ Search connection: {connection.name} (ID: {connection.id})")

### Create an agent grounded in your search index

We define the agent with an `AzureAISearchAgentTool` that points at our index. The agent will automatically call this tool to retrieve context before generating answers.

In [ ]:
agent = project_client.agents.create_version(
    agent_name="earth-at-night-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=(
            "You are a helpful assistant that answers questions about Earth at night. "
            "Always cite your sources using [doc_id†source] notation."
        ),
        tools=[
            AzureAISearchAgentTool(
                azure_ai_search=AzureAISearchToolResource(
                    indexes=[
                        AISearchIndexResource(
                            project_connection_id=connection.id,
                            index_name=INDEX_NAME,
                            query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
                        )
                    ]
                )
            )
        ],
    ),
    description="Agent grounded in NASA Earth at Night data via Foundry IQ",
)

print(f"✅ Agent created (name={agent.name}, version={agent.version})")

### Ask the agent a question

We use the OpenAI-compatible Responses API to query the agent. The agent automatically retrieves relevant content from the search index and returns a cited answer.

In [ ]:
openai_client = project_client.get_openai_client()

response = openai_client.responses.create(
    input="Why is the Phoenix nighttime street grid so sharply visible from space?",
    tool_choice="required",
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

print("📝 Agent response:\n")
print(response.output_text)

The agent used the search tool behind the scenes, retrieved matching passages, and wove them into a cited response — no manual retrieval code required.

### Clean up the agent

In [ ]:
project_client.agents.delete_version(
    agent_name=agent.name, agent_version=agent.version
)
print("✅ Agent deleted")

---

## Clean Up

Remove the Foundry IQ resources we created. If you want to keep experimenting, skip this cell.

In [ ]:
index_client.delete_knowledge_base(KNOWLEDGE_BASE_NAME)
print(f"Deleted knowledge base '{KNOWLEDGE_BASE_NAME}'")

index_client.delete_knowledge_source(knowledge_source=KNOWLEDGE_SOURCE_NAME)
print(f"Deleted knowledge source '{KNOWLEDGE_SOURCE_NAME}'")

index_client.delete_index(INDEX_NAME)
print(f"Deleted index '{INDEX_NAME}'")

---

## Conclusion

In this cookbook we walked through the complete Foundry IQ lifecycle:

1. **Knowledge source** — We created a pointer to a search index so Foundry IQ knows *where* to look.
2. **Knowledge base** — We paired the knowledge source with an LLM, enabling query planning and answer synthesis.
3. **Agentic retrieval** — We queried the knowledge base and saw how it decomposes questions, retrieves in parallel, and returns cited answers.
4. **Agent Service integration** — We embedded the same search index into a Foundry agent, letting it ground responses automatically without manual retrieval code.

### Where to go next

- **Add more knowledge sources** — A single knowledge base can span multiple indexes and data types. Try adding a second index with different content.
- **Tune retrieval** — Experiment with `retrieval_reasoning_effort` levels and `answer_instructions` to control cost and quality.
- **Evaluate quality** — Use the [Azure AI Evaluation SDK](https://learn.microsoft.com/azure/ai-foundry/how-to/develop/evaluate-sdk) to measure groundedness and relevance at scale.
- **Explore the IQ Series** — Continue learning with [Episode 2: Data Pipelines](../../2-Foundry-IQ-Building-the-Data-Pipeline-with-Knowledge-Sources/) and [Episode 3: Multi-Source Queries](../../3-Foundry-IQ-Querying-the-Multi-Source-AI-Knowledge-Bases/).